# GTS Spike: Validate Expression Tree Decomposition

**Goal:** Test if GTS can decompose English math word problems into parseable expression trees.

**Steps:**
1. Install MWPToolkit
2. Train GTS on MAWPS (English dataset)
3. Run inference on test problems
4. Parse prefix output to expression tree
5. Check depth and atomicity

**Runtime:** Select GPU: Runtime → Change runtime type → T4 GPU

In [ ]:
# Check GPU is available
!nvidia-smi

## 1. Install MWPToolkit

In [ ]:
# Clone MWPToolkit repo
!git clone https://github.com/LYH-YF/MWPToolkit.git
%cd MWPToolkit

In [ ]:
# Install dependencies
!pip install -r requirements.txt
!pip install stanza  # For NLP preprocessing

In [ ]:
# Download stanza English model for tokenization
import stanza
stanza.download('en')

## 2. Check Available Datasets

In [ ]:
# List available datasets
!ls dataset/

In [ ]:
# Check MAWPS dataset (English)
import json

# Try to find MAWPS data
!find . -name "*mawps*" -o -name "*MAWPS*" 2>/dev/null | head -20

In [ ]:
# Look at a sample problem from available dataset
import os
import json

# Check what datasets exist
for dataset_dir in os.listdir('dataset'):
    print(f"Dataset: {dataset_dir}")
    dataset_path = f'dataset/{dataset_dir}'
    if os.path.isdir(dataset_path):
        for f in os.listdir(dataset_path)[:3]:
            print(f"  - {f}")

## 3. Train GTS Model

This will take ~1-2 hours on T4 GPU for MAWPS dataset.

In [ ]:
# Train GTS on MAWPS (or mawps_single if that's the name)
# Adjust dataset name based on what's available

!python run_mwptoolkit.py \
    --model=GTS \
    --dataset=mawps \
    --task_type=single_equation \
    --equation_fix=prefix \
    --k_fold=None \
    --gpu_id=0 \
    --epoch=80

## 4. Test Inference

If training completed, let's test the model on some problems.

In [ ]:
# Find the trained model
!ls -la trained_model/

In [ ]:
# Load trained model for inference
# This may need adjustment based on MWPToolkit's API

import torch
import sys
sys.path.insert(0, '.')

from mwptoolkit.quick_start import run_toolkit

# Check the config used
!cat trained_model/GTS_mawps/config.json 2>/dev/null || echo "Config not found, checking other locations..."
!find trained_model -name "*.json" | head -5

In [ ]:
# Manual inference test
# We'll need to adapt this based on MWPToolkit's actual inference API

test_problems = [
    "John has 5 apples. Mary gives him 3 more apples. How many apples does John have now?",
    "A store has 24 books. They sell 8 books. How many books are left?",
    "Tom has 10 dollars. He buys a toy for 3 dollars and a book for 4 dollars. How much money does he have left?",
    "There are 15 birds in a tree. 7 fly away and then 4 more arrive. How many birds are in the tree?",
    "A farmer has 36 eggs. He puts them equally into 6 baskets. How many eggs are in each basket?"
]

print("Test problems:")
for i, p in enumerate(test_problems):
    print(f"{i+1}. {p}")

## 5. Parse Prefix Output to Expression Tree

In [ ]:
# Expression tree parser (same as our planned integration)

from dataclasses import dataclass
from typing import Optional
from enum import Enum

class NodeType(Enum):
    OPERATOR = "operator"
    NUMBER = "number"
    VARIABLE = "variable"

@dataclass
class ExprNode:
    type: NodeType
    value: str
    left: Optional["ExprNode"] = None
    right: Optional["ExprNode"] = None
    
    @property
    def depth(self) -> int:
        if self.type != NodeType.OPERATOR:
            return 0
        left_depth = self.left.depth if self.left else 0
        right_depth = self.right.depth if self.right else 0
        return 1 + max(left_depth, right_depth)
    
    @property
    def is_atomic(self) -> bool:
        return self.depth <= 1
    
    def __repr__(self):
        if self.type != NodeType.OPERATOR:
            return self.value
        return f"({self.value} {self.left} {self.right})"

def parse_prefix(prefix_str: str) -> ExprNode:
    """Parse prefix notation to expression tree."""
    tokens = prefix_str.replace('(', '').replace(')', '').split()
    idx = 0
    
    def parse() -> ExprNode:
        nonlocal idx
        if idx >= len(tokens):
            raise ValueError("Unexpected end of expression")
        
        token = tokens[idx]
        idx += 1
        
        # Operators
        if token in ['+', '-', '*', '/', '^', 'ADD', 'SUB', 'MUL', 'DIV']:
            # Normalize operator names
            op_map = {'ADD': '+', 'SUB': '-', 'MUL': '*', 'DIV': '/'}
            op = op_map.get(token, token)
            return ExprNode(
                type=NodeType.OPERATOR,
                value=op,
                left=parse(),
                right=parse()
            )
        # Variables (N0, N1, etc. or temp_a, temp_b)
        elif token.startswith('N') or token.startswith('temp_') or token.startswith('num'):
            return ExprNode(type=NodeType.VARIABLE, value=token)
        # Numbers
        else:
            try:
                float(token)  # Verify it's a number
                return ExprNode(type=NodeType.NUMBER, value=token)
            except ValueError:
                # Unknown token, treat as variable
                return ExprNode(type=NodeType.VARIABLE, value=token)
    
    return parse()

# Test the parser with example prefix expressions
test_prefixes = [
    "+ 5 3",           # 5 + 3, depth 1
    "- + 5 3 2",       # (5 + 3) - 2, depth 2
    "+ - 10 3 4",      # (10 - 3) + 4, depth 2
    "/ 36 6",          # 36 / 6, depth 1
]

print("Testing prefix parser:")
print("=" * 50)
for prefix in test_prefixes:
    tree = parse_prefix(prefix)
    print(f"Prefix: {prefix}")
    print(f"Tree:   {tree}")
    print(f"Depth:  {tree.depth}")
    print(f"Atomic: {tree.is_atomic}")
    print()

## 6. Recursive Decomposition

In [ ]:
def decompose_to_atomic(tree: ExprNode, step_num: int = 1) -> list[tuple[int, str, ExprNode]]:
    """
    Recursively decompose tree into atomic steps.
    Returns list of (step_number, operation_description, atomic_tree)
    """
    if tree.is_atomic:
        op_names = {'+': 'add', '-': 'subtract', '*': 'multiply', '/': 'divide'}
        op_desc = f"{op_names.get(tree.value, tree.value)} two numbers"
        return [(step_num, op_desc, tree)]
    
    steps = []
    current_step = step_num
    
    # Process left subtree if not atomic
    if tree.left and not tree.left.is_atomic:
        left_steps = decompose_to_atomic(tree.left, current_step)
        steps.extend(left_steps)
        current_step += len(left_steps)
        # Replace with step reference
        tree.left = ExprNode(type=NodeType.VARIABLE, value=f"step_{current_step - 1}")
    
    # Process right subtree if not atomic
    if tree.right and not tree.right.is_atomic:
        right_steps = decompose_to_atomic(tree.right, current_step)
        steps.extend(right_steps)
        current_step += len(right_steps)
        # Replace with step reference
        tree.right = ExprNode(type=NodeType.VARIABLE, value=f"step_{current_step - 1}")
    
    # Now tree should be atomic
    steps.extend(decompose_to_atomic(tree, current_step))
    
    return steps

# Test decomposition
print("Testing recursive decomposition:")
print("=" * 50)

# Complex expression: (5 + 3) - 2 = "- + 5 3 2"
complex_prefix = "- + 5 3 2"
tree = parse_prefix(complex_prefix)
print(f"Original: {complex_prefix}")
print(f"Tree:     {tree}")
print(f"Depth:    {tree.depth}")
print()

# Need to re-parse since decompose modifies tree
tree = parse_prefix(complex_prefix)
steps = decompose_to_atomic(tree)
print("Decomposed steps:")
for step_num, op_desc, atomic_tree in steps:
    print(f"  Step {step_num}: {op_desc}")
    print(f"          Tree: {atomic_tree}")
    print(f"          Depth: {atomic_tree.depth} (atomic: {atomic_tree.is_atomic})")

## 7. Results Summary

Fill this in after running the notebook:

In [ ]:
# Summary template - fill in after testing

results = {
    "installation": "SUCCESS / FAILED",
    "training_time": "X hours",
    "training_accuracy": "X%",
    "test_problems_tried": 0,
    "correct_parses": 0,
    "issues_encountered": [],
    "recommendation": "PROCEED / PIVOT / NEEDS_MORE_TESTING"
}

print("=" * 50)
print("GTS SPIKE RESULTS")
print("=" * 50)
for k, v in results.items():
    print(f"{k}: {v}")

## 8. Save Trained Model (if successful)

Download the trained model to use locally.

In [ ]:
# Zip trained model for download
!zip -r gts_mawps_trained.zip trained_model/

# In Colab, this will let you download
from google.colab import files
files.download('gts_mawps_trained.zip')